# RF-DETR-XL · AI Hub 알약 객체 탐지 (v2: SAM · SimCLR 스위치)

10,000장 내외 / 118종 조합 데이터용 전체 파이프라인입니다. **RTX 5060(8GB, Blackwell) 기준**으로 정리했습니다.

- AI Hub의 partial JSON을 이미지 단위로 병합하고, `images[0].dl_mapping_code`를 실제 약품 클래스로 사용합니다.
- flip·색상 파생본이 train/valid에 갈라지지 않도록 **augmentation family 단위**로 분할합니다.
- RF-DETR이 자동 인식하는 COCO 폴더를 만들고 RF-DETR-XL을 fine-tuning합니다.
- 추가 기능: `USE_SAM`(박스 정제), `USE_SIMCLR`(crop 분류기 재판정)을 설정 셀에서 True/False로 선택합니다.

> RF-DETR-XL은 `rfdetr_plus` 패키지 모델이며 코드/가중치는 PML-1.0 적용 대상입니다.
> 아래 공식 라이선스를 검토하고 동의한 경우에만 `ACCEPT_PLATFORM_MODEL_LICENSE = True`로 바꾸세요(기본값 False).
> <https://github.com/roboflow/rf-detr_plus/blob/main/LICENSE>


## v2 추가 기능 요약

`USE_SAM`, `USE_SIMCLR` 스위치(설정 셀)로 아래 기능을 독립적으로 켜고 끕니다. 둘 다 `False`면 기존 파이프라인과 같은 결과를 냅니다.

| 스위치 | 하는 일 | 학습 데이터 |
|---|---|---|
| `USE_SAM` | 검출 박스를 SAM box prompt로 넣어 마스크 외접 박스로 정제 (학습 없음) | 없음 |
| `USE_SIMCLR` | train GT crop으로 SimCLR 대조학습 → 118종 분류기 → 검출 클래스 재판정·융합 | train split만 |

- 실행 순서: 검출기 학습 → best 로드 → **val 검출 캐시 → SAM → SimCLR → 변형 비교표 → 제출**
- 비교표는 `base / sam / simclr / sam+simclr` 중 켜진 조합만 val `mAP@[.75:.95]`로 계산합니다.
- SAM·SimCLR 준비가 실패하면 오류를 출력한 뒤 해당 기능만 끄고 계속 진행합니다.
- `SUBMIT_USE_BEST_VARIANT=True`면 스위치와 무관하게 val 점수가 가장 높은 변형으로 제출합니다.
- SAM 박스 padding, SimCLR 융합 비율은 val로 스칼라 1개씩 자동 튜닝합니다(`SAM_TUNE_PAD`, `SIMCLR_TUNE_ALPHA`).


## 0. 설치

아래 셀을 한 번 실행한 뒤 **커널을 재시작**하고 1번 셀부터 실행하세요.
RTX 5060(sm_120)은 **CUDA 12.8 이상(cu128) · PyTorch 2.7 이상**이 필요합니다. imports 셀의 GPU 호환성 검사에서 오류가 나면 복구 셀을 실행하세요.


In [ ]:
# imports 셀에서 "sm_120 미지원" 오류가 날 때만 아래 두 줄의 주석을 풀어 실행하세요. (실행 후 Kernel Restart)
# %pip uninstall -y torch torchvision torchaudio
# %pip install --no-cache-dir torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128


In [ ]:
%pip install -q -U "rfdetr[plus,train,loggers]" kagglehub iterative-stratification "torchmetrics[detection]" faster-coco-eval wandb pycocotools supervision pandas matplotlib
%pip install -q -U ultralytics   # SAM(USE_SAM)용. 이미 torch가 있으면 torch는 바꾸지 않습니다.


## 1. imports · 재현성 · 하드웨어 확인


In [ ]:
import contextlib
import hashlib
import gc
import json
import math
import os
import platform
import random
import re
import shutil
import sys
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import supervision as sv
import torch
import wandb
from IPython.display import display
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from PIL import Image
from pycocotools.coco import COCO
from rfdetr_plus import RFDETRXLarge
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from tqdm.auto import tqdm


SEED = 42


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
torch.set_float32_matmul_precision("high")

print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"OS     : {platform.platform()}")
print(f"CUDA   : {torch.cuda.is_available()}")
def check_gpu_compat():
    """RTX 50 시리즈(Blackwell, sm_120)는 CUDA 12.8 이상 빌드의 PyTorch가 필요합니다."""
    if not torch.cuda.is_available():
        return
    major, minor = torch.cuda.get_device_capability(0)
    if major >= 12 and f"sm_{major}{minor}" not in torch.cuda.get_arch_list():
        raise RuntimeError(
            f"이 PyTorch({torch.__version__}, CUDA {torch.version.cuda})는 GPU 아키텍처 sm_{major}{minor}를 지원하지 않습니다.\n"
            "설치 섹션의 cu128 복구 셀을 실행한 뒤 커널을 재시작하세요."
        )

check_gpu_compat()

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU    : {gpu.name}")
    print(f"VRAM   : {gpu.total_memory / 1024**3:.1f} GiB")
else:
    raise RuntimeError("RF-DETR-XL 학습에는 CUDA GPU 사용을 강하게 권장합니다.")


## 2. 실험 설정

RF-DETR-XL의 pretrained native resolution은 700이지만, RTX 5060 8GB에서는 **560 / AutoBatch / gradient checkpointing**으로 시작합니다.
AutoBatch가 물리 배치와 gradient accumulation을 자동으로 조정합니다.


In [ ]:
# 데이터 검증 기대값
EXPECTED_NUM_IMAGES = None   # None이면 이미지 수 고정 검증을 하지 않음(실제 수는 출력됨)
EXPECTED_NUM_CLASSES = 118
EXPECTED_TEST_IMAGES = 842

# 분할
VAL_RATIO = 0.20
N_SPLIT_CANDIDATES = 200
GROUP_GEOMETRY_DECIMALS = 3
FAMILY_STRICT = False   # True: family 탐지가 이상하면 중단 / False: 경고만 출력하고 진행

# RF-DETR-XL 학습
RESOLUTION = 560          # RTX 5060 8GB에서 RF-DETR-XL을 안정적으로 학습할 해상도
EPOCHS = 30
TRAIN_BATCH = "auto"          # GPU 메모리를 실제로 probe
EVAL_BATCH_SIZE = 1
AUTO_BATCH_EFFECTIVE = 8      # 보통 physical batch 1 + accumulation으로 동작
LR = 1e-4
LR_ENCODER = 1.5e-4
WEIGHT_DECAY = 1e-4
GRADIENT_CHECKPOINTING = True # 메모리 약 30~40% 절감, 학습은 다소 느려짐
EARLY_STOPPING = False        # 정확히 30 epoch 비교; 필요하면 True로 변경

# Windows/Jupyter에서는 worker subprocess 문제가 잦으므로 0이 안전합니다.
IS_WINDOWS = platform.system() == "Windows"
NUM_WORKERS = 0 if IS_WINDOWS else min(4, os.cpu_count() or 1)
PIN_MEMORY = not IS_WINDOWS
PERSISTENT_WORKERS = False

# 로깅/출력
USE_WANDB = False  # W&B 권한 오류가 학습을 막지 않게 기본은 로컬 기록
WANDB_ENTITY = "jaedong0817-"
WANDB_PROJECT = "beginner-team-project"
RUN_NAME = f"rfdetr_xlarge_pills_r{RESOLUTION}_e{EPOCHS}_rtx5060"
OUTPUT_DIR = Path(WANDB_PROJECT) / RUN_NAME
SUBMISSION_TAG = "rfdetr_xlarge"   # 제출 파일명 접두어
RUN_TRAINING = True
RESUME_TRAINING = False  # Run All에서 중복 학습되지 않도록 기본 False
if RUN_TRAINING and RESUME_TRAINING:
    raise ValueError("RUN_TRAINING과 RESUME_TRAINING은 동시에 True일 수 없습니다.")

# 예측/평가
PRED_BATCH_SIZE = 1            # VRAM 여유가 확인되면 2로 올려도 됨
PRED_CONFIDENCE = 0.001        # AP/제출 후보 보존용 낮은 threshold
EVAL_MAX_DETECTIONS = 20
SUBMISSION_MAX_DETECTIONS = 8 # 실제 객체 수 3~4개에 여유
COMPETITION_IOUS = [0.75, 0.80, 0.85, 0.90, 0.95]
assert EVAL_MAX_DETECTIONS >= SUBMISSION_MAX_DETECTIONS

# PML-1.0을 읽고 동의한 뒤에만 True로 바꾸세요.
ACCEPT_PLATFORM_MODEL_LICENSE = False

# ══════════════════════════════════════════════════════════════
# 추가 기능 스위치 (v2)
# ══════════════════════════════════════════════════════════════
USE_SAM = False      # True: SAM으로 검출 박스를 마스크 외접 박스로 정제
USE_SIMCLR = False   # True: SimCLR 사전학습 crop 분류기로 클래스 재판정 · 융합
SUBMIT_USE_BEST_VARIANT = False  # True: 스위치와 무관하게 val 점수가 가장 높은 변형으로 제출

# ── SAM 세부 설정 ─────────────────────────────────────────────
SAM_MODEL_CANDIDATES = ["sam2.1_b.pt", "sam2.1_l.pt", "sam_b.pt", "mobile_sam.pt"]  # 앞에서부터 로드 시도
SAM_TOPK = 8                # 이미지당 상위 몇 개 검출을 정제할지
SAM_MIN_SCORE = 0.01        # 이 점수 미만 검출은 정제하지 않음
SAM_MIN_IOU = 0.60          # 원본 박스와 IoU가 이보다 낮은 SAM 결과는 버리고 원본 유지
SAM_MIN_AREA_RATIO = 0.30   # 마스크 면적 / 박스 면적이 이보다 작으면 버림
SAM_BLEND = 1.0             # 1.0 = SAM 박스로 완전 교체, 0.5 = 원본과 평균
SAM_PAD_PX = 0.0            # 마스크 박스를 사방으로 넓히는 픽셀 수(GT 박스가 느슨하면 양수가 유리)
SAM_TUNE_PAD = True         # True: val에서 SAM_PAD_GRID를 비교해 SAM_PAD_PX 자동 선택
SAM_PAD_GRID = (-2.0, -1.0, 0.0, 1.0, 2.0, 3.0, 4.0, 6.0)

# ── SimCLR 세부 설정 (GT crop → 대조학습 → 118종 분류기 → 검출 클래스 재판정) ──
SIMCLR_BACKBONE = "resnet50"      # resnet18 / resnet34 / resnet50
SIMCLR_IMAGENET_INIT = True       # True: ImageNet 가중치에서 시작(다운로드 실패 시 자동으로 랜덤 초기화)
SIMCLR_CROP = 160                 # crop 한 변(px)
SIMCLR_CROP_PAD = 0.15            # 박스 주변 여백 비율
SIMCLR_EPOCHS = 30                # 대조학습 epoch (0이면 대조학습 생략, 분류 fine-tuning만)
SIMCLR_BATCH = 128                # OOM이면 자동으로 절반씩 줄여 재시도
SIMCLR_LR = 5e-4
SIMCLR_TEMP = 0.2
SIMCLR_CLS_EPOCHS = 12            # 118종 분류 fine-tuning epoch
SIMCLR_CLS_LR_BACKBONE = 2e-4
SIMCLR_CLS_LR_HEAD = 2e-3
SIMCLR_TOPK = 8                   # 이미지당 상위 몇 개 검출에 분류기를 적용할지
SIMCLR_MIN_SCORE = 0.01
SIMCLR_TOPN = 2                   # 검출 하나당 출력할 클래스 후보 수
SIMCLR_FUSE_ALPHA = 0.5           # 0=검출기 클래스만, 1=분류기 확률만. q = (1-a)*onehot(검출클래스) + a*분류확률
SIMCLR_TUNE_ALPHA = True          # True: val에서 SIMCLR_ALPHA_GRID를 비교해 alpha 자동 선택
SIMCLR_ALPHA_GRID = (0.25, 0.5, 0.75, 1.0)
SIMCLR_TTA = True                 # 0/90/180/270도 회전 평균
SIMCLR_REUSE_CACHE = True         # 같은 split·설정으로 학습한 가중치가 있으면 재사용

print({
    "resolution": RESOLUTION,
    "epochs": EPOCHS,
    "batch": TRAIN_BATCH,
    "gradient_checkpointing": GRADIENT_CHECKPOINTING,
    "workers": NUM_WORKERS,
    "run_training": RUN_TRAINING,
    "resume_training": RESUME_TRAINING,
    "output_dir": str(OUTPUT_DIR),
})


## 3. 데이터 경로

Kaggle competition 다운로드 또는 로컬 경로 중 하나를 사용합니다.


In [ ]:
DOWNLOAD_FROM_KAGGLE = True
KAGGLE_COMPETITION = "ai14-level-project"
LOCAL_DATA_ROOT = Path(r"D:\path\to\sprint_ai_project1_data")

if DOWNLOAD_FROM_KAGGLE:
    kagglehub.login()
    downloaded_path = Path(kagglehub.competition_download(KAGGLE_COMPETITION))
    nested_path = downloaded_path / "sprint_ai_project1_data"
    DATA_ROOT = nested_path if nested_path.exists() else downloaded_path
else:
    DATA_ROOT = LOCAL_DATA_ROOT.expanduser()

TRAIN_IMAGE_DIR = DATA_ROOT / "train_images"
TRAIN_ANNOTATION_DIR = DATA_ROOT / "train_annotations"
TEST_IMAGE_DIR = DATA_ROOT / "test_images"

for required_dir in (TRAIN_IMAGE_DIR, TRAIN_ANNOTATION_DIR, TEST_IMAGE_DIR):
    if not required_dir.is_dir():
        raise FileNotFoundError(f"필수 폴더가 없습니다: {required_dir}")

IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}


def list_images(directory: Path) -> list[Path]:
    return sorted(
        p for p in Path(directory).iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES
    )


train_image_paths = list_images(TRAIN_IMAGE_DIR)
test_paths = list_images(TEST_IMAGE_DIR)
annotation_paths = sorted(TRAIN_ANNOTATION_DIR.rglob("*.json"))

print(f"DATA_ROOT       : {DATA_ROOT}")
print(f"Train 이미지 수: {len(train_image_paths):,}")
print(f"Annotation 수   : {len(annotation_paths):,}")
print(f"Test 이미지 수 : {len(test_paths):,}")

if EXPECTED_NUM_IMAGES is not None and len(train_image_paths) != EXPECTED_NUM_IMAGES:
    raise RuntimeError(
        f"Train 이미지 수 불일치: 예상={EXPECTED_NUM_IMAGES:,}, 실제={len(train_image_paths):,}"
    )
if EXPECTED_TEST_IMAGES is not None and len(test_paths) != EXPECTED_TEST_IMAGES:
    raise RuntimeError(
        f"Test 이미지 수 불일치: 예상={EXPECTED_TEST_IMAGES:,}, 실제={len(test_paths):,}"
    )


## 4. AI Hub JSON 보정 · partial annotation 병합

한 이미지에 여러 JSON이 있을 수 있습니다. 각 JSON의 `images[0].dl_mapping_code`가
그 JSON bbox의 실제 약품 클래스입니다. 원본 JSON의 generic `category_id=1`은 사용하지 않습니다.


In [ ]:
image_stem_to_path = {p.stem: p for p in train_image_paths}
if len(image_stem_to_path) != len(train_image_paths):
    raise RuntimeError("서로 다른 확장자에 동일 stem을 가진 train 이미지가 있습니다.")

stem_to_annotation_paths = defaultdict(list)
for annotation_path in annotation_paths:
    stem_to_annotation_paths[annotation_path.stem].append(annotation_path)

valid_stems = sorted(set(image_stem_to_path) & set(stem_to_annotation_paths))
images_without_annotation = set(image_stem_to_path) - set(stem_to_annotation_paths)
annotations_without_image = set(stem_to_annotation_paths) - set(image_stem_to_path)

print(f"매핑 가능한 이미지         : {len(valid_stems):,}")
print(f"annotation 없는 이미지     : {len(images_without_annotation):,}")
print(f"이미지 없는 annotation stem: {len(annotations_without_image):,}")

if EXPECTED_NUM_IMAGES is not None and len(valid_stems) != EXPECTED_NUM_IMAGES:
    raise RuntimeError(
        "모든 train 이미지가 annotation과 매핑되지 않았습니다. 위 누락 목록의 원인을 먼저 확인하세요."
    )


def load_aihub_json(json_path: Path) -> dict:
    """AI Hub의 잘못된 `\"area\": ,`만 null로 최소 보정해 읽습니다."""
    text = Path(json_path).read_text(encoding="utf-8")
    text = re.sub(r'("area"\s*:\s*),', r'\1null,', text)
    try:
        return json.loads(text)
    except json.JSONDecodeError as exc:
        raise RuntimeError(
            f"JSON 파싱 실패: {json_path}\n"
            f"line={exc.lineno}, column={exc.colno}: {exc.msg}"
        ) from exc


def clip_xywh(bbox, image_w: int, image_h: int):
    try:
        x, y, bw, bh = map(float, bbox)
    except (TypeError, ValueError):
        return None
    if not np.isfinite([x, y, bw, bh]).all() or bw <= 0 or bh <= 0:
        return None
    x1 = min(max(x, 0.0), float(image_w))
    y1 = min(max(y, 0.0), float(image_h))
    x2 = min(max(x + bw, 0.0), float(image_w))
    y2 = min(max(y + bh, 0.0), float(image_h))
    if x2 <= x1 or y2 <= y1:
        return None
    return [x1, y1, x2 - x1, y2 - y1]


def build_image_records(stems, grouped_paths):
    image_records = {}
    code_to_name = {}
    stats = Counter()

    for stem in tqdm(stems, desc="AI Hub JSON 병합"):
        source_path = image_stem_to_path[stem]
        with Image.open(source_path) as image:
            actual_w, actual_h = image.size

        annotations = []
        seen_annotations = set()

        for annotation_path in grouped_paths[stem]:
            data = load_aihub_json(annotation_path)
            if not data.get("images"):
                raise ValueError(f"images 정보가 없는 annotation: {annotation_path}")

            image_meta = data["images"][0]
            drug_code = str(image_meta["dl_mapping_code"]).strip()
            drug_name = str(image_meta["dl_name"]).strip()
            meta_size = (int(image_meta["width"]), int(image_meta["height"]))
            if meta_size != (actual_w, actual_h):
                raise ValueError(
                    f"실제 이미지/JSON 크기 불일치: {source_path.name}, "
                    f"actual={(actual_w, actual_h)}, json={meta_size}"
                )

            old_name = code_to_name.get(drug_code)
            if old_name is not None and old_name != drug_name:
                raise ValueError(
                    f"같은 dl_mapping_code의 이름 불일치: {drug_code} -> "
                    f"{old_name!r} / {drug_name!r}"
                )

            for ann in data.get("annotations", []):
                bbox = ann.get("bbox", [])
                if not isinstance(bbox, (list, tuple)) or len(bbox) != 4:
                    stats["invalid_bbox"] += 1
                    continue

                clipped = clip_xywh(bbox, actual_w, actual_h)
                if clipped is None:
                    stats["invalid_bbox"] += 1
                    continue

                x, y, bw, bh = clipped
                dedup_key = (
                    drug_code,
                    round(x, 4), round(y, 4), round(bw, 4), round(bh, 4),
                )
                if dedup_key in seen_annotations:
                    stats["duplicate_bbox"] += 1
                    continue
                seen_annotations.add(dedup_key)

                raw_area = ann.get("area")
                try:
                    raw_area = float(raw_area)
                    if not math.isfinite(raw_area) or raw_area <= 0:
                        raise ValueError
                except (TypeError, ValueError):
                    stats["repaired_area"] += 1

                annotations.append({
                    "bbox": [x, y, bw, bh],
                    "category_code": drug_code,
                    "area": bw * bh,  # clip 이후 bbox와 항상 일치
                    "iscrowd": 0,
                })
                code_to_name[drug_code] = drug_name

        if not annotations:
            stats["empty_images"] += 1
            continue

        image_records[stem] = {
            "image_path": str(source_path),
            "file_name": source_path.name,
            "width": actual_w,
            "height": actual_h,
            "annotations": annotations,
        }

    return image_records, code_to_name, stats


image_records, code_to_name, merge_stats = build_image_records(
    valid_stems, stem_to_annotation_paths
)

print(f"최종 image record: {len(image_records):,}")
print(f"실제 약품 클래스 : {len(code_to_name):,}")
print(f"병합 통계        : {dict(merge_stats)}")

if EXPECTED_NUM_IMAGES is not None and len(image_records) != EXPECTED_NUM_IMAGES:
    raise RuntimeError(
        f"유효 record 수 불일치: 예상={EXPECTED_NUM_IMAGES:,}, 실제={len(image_records):,}"
    )


## 5. 0-based 모델 클래스 · 제출 category 매핑


In [ ]:
sorted_codes = sorted(code_to_name)
code_to_model_id = {code: idx for idx, code in enumerate(sorted_codes)}
model_id_to_code = {idx: code for code, idx in code_to_model_id.items()}
model_id_to_name = {idx: code_to_name[code] for idx, code in model_id_to_code.items()}


def dl_code_to_submission_id(class_code: str) -> int:
    match = re.fullmatch(r"K-(\d+)", str(class_code).strip())
    if match is None:
        raise ValueError(f"예상하지 못한 dl_mapping_code: {class_code!r}")
    return int(match.group(1))


code_to_submission_id = {
    code: dl_code_to_submission_id(code) for code in sorted_codes
}

num_classes = len(sorted_codes)
if num_classes != EXPECTED_NUM_CLASSES:
    raise RuntimeError(
        f"클래스 수 불일치: 예상={EXPECTED_NUM_CLASSES}, 실제={num_classes}. "
        "annotation 폴더와 병합 결과를 확인하세요."
    )

assert set(model_id_to_code) == set(range(num_classes))
assert len(set(code_to_submission_id.values())) == num_classes

mapping_preview = pd.DataFrame([
    {
        "model_id": model_id,
        "dl_mapping_code": model_id_to_code[model_id],
        "submission_category_id": code_to_submission_id[model_id_to_code[model_id]],
        "name": model_id_to_name[model_id],
    }
    for model_id in range(num_classes)
])
display(mapping_preview.head(10))


## 6. augmentation-family 그룹 분할

색상/채도 변형은 bbox가 같고, H/V flip은 bbox 중심이 `(x, y) → (1-x, 1-y)`로 바뀝니다.
각 bbox 중심을 flip 불변 형태인 `min(c, 1-c)`로 바꾼 뒤 클래스·크기와 함께 hash하여
파생본 family를 찾습니다. 이후 **family 전체를 하나의 분할에만** 넣습니다.

이 셀은 family 탐지가 데이터 특성과 맞지 않으면 조용히 넘어가지 않고 중단합니다.


In [ ]:
def augmentation_family_signature(record: dict, decimals: int = 3) -> str:
    width = float(record["width"])
    height = float(record["height"])
    tokens = []
    for ann in record["annotations"]:
        x, y, bw, bh = ann["bbox"]
        cx = (x + bw / 2) / width
        cy = (y + bh / 2) / height
        token = (
            ann["category_code"],
            round(min(cx, 1.0 - cx), decimals),
            round(min(cy, 1.0 - cy), decimals),
            round(bw / width, decimals),
            round(bh / height, decimals),
        )
        tokens.append(token)
    payload = json.dumps(sorted(tokens), ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:20]


def family_warn_or_raise(message: str) -> None:
    """FAMILY_STRICT=True면 중단, False면 경고만 출력하고 계속 진행합니다."""
    if FAMILY_STRICT:
        raise RuntimeError(message)
    print("⚠ 경고:", message)


family_to_stems = defaultdict(list)
for stem, record in image_records.items():
    signature = augmentation_family_signature(record, GROUP_GEOMETRY_DECIMALS)
    family_to_stems[signature].append(stem)

group_sizes = Counter(len(stems) for stems in family_to_stems.values())
singleton_ratio = group_sizes.get(1, 0) / len(family_to_stems)
largest_group = max(map(len, family_to_stems.values()))

print(f"family 수          : {len(family_to_stems):,}")
print(f"family 크기 분포   : {dict(sorted(group_sizes.items()))}")
print(f"singleton family 비율: {singleton_ratio:.1%}")
print(f"최대 family 크기   : {largest_group}")

if singleton_ratio > 0.50:
    family_warn_or_raise(
        "자동 family 탐지 결과 singleton이 50%를 넘습니다. 파생본이 거의 탐지되지 않았거나 데이터에 파생본이 적다는 뜻입니다. "
        "파일명 규칙에 원본 ID가 있다면 augmentation_family_signature()를 그 ID 추출 방식으로 바꾸세요."
    )
if largest_group > 8:
    family_warn_or_raise(
        f"family 크기 {largest_group}이 비정상적으로 큽니다. 서로 다른 원본이 같은 geometry signature로 충돌했는지 확인하세요."
    )


In [ ]:
family_ids = sorted(family_to_stems)
family_labels = []
family_weights = []

for family_id in family_ids:
    stems = family_to_stems[family_id]
    present_codes = {
        ann["category_code"]
        for stem in stems
        for ann in image_records[stem]["annotations"]
    }
    row = np.zeros(num_classes, dtype=np.uint8)
    for code in present_codes:
        row[code_to_model_id[code]] = 1
    family_labels.append(row)
    family_weights.append(len(stems))

family_labels = np.asarray(family_labels)
family_weights = np.asarray(family_weights)
overall_class_images = np.zeros(num_classes, dtype=np.int64)
for record in image_records.values():
    for code in {ann["category_code"] for ann in record["annotations"]}:
        overall_class_images[code_to_model_id[code]] += 1

# 하나의 family에만 존재하는 클래스는 그 family를 반드시 train에 둡니다.
class_family_support = family_labels.sum(axis=0)
forced_train_family_idx = set(
    np.flatnonzero(family_labels[:, class_family_support == 1].any(axis=1)).tolist()
)
free_family_idx = np.asarray(
    [i for i in range(len(family_ids)) if i not in forced_train_family_idx],
    dtype=int,
)
desired_valid_images = round(len(image_records) * VAL_RATIO)
free_image_count = int(family_weights[free_family_idx].sum())
free_valid_ratio = desired_valid_images / free_image_count
if not 0 < free_valid_ratio < 1:
    raise RuntimeError(
        f"강제 train family 제외 후 valid 비율을 만들 수 없습니다: {free_valid_ratio:.3f}"
    )

print(f"희소 클래스 때문에 train 고정된 family: {len(forced_train_family_idx)}")


def split_score(train_family_idx, valid_family_idx):
    train_stems_ = [
        stem for idx in train_family_idx for stem in family_to_stems[family_ids[idx]]
    ]
    valid_stems_ = [
        stem for idx in valid_family_idx for stem in family_to_stems[family_ids[idx]]
    ]

    train_counts = np.zeros(num_classes, dtype=np.int64)
    valid_counts = np.zeros(num_classes, dtype=np.int64)
    for stem in train_stems_:
        for code in {a["category_code"] for a in image_records[stem]["annotations"]}:
            train_counts[code_to_model_id[code]] += 1
    for stem in valid_stems_:
        for code in {a["category_code"] for a in image_records[stem]["annotations"]}:
            valid_counts[code_to_model_id[code]] += 1

    missing_train = int((train_counts == 0).sum())
    missing_valid = int((valid_counts == 0).sum())
    ratio_error = abs(len(valid_stems_) / len(image_records) - VAL_RATIO)
    valid_fraction = np.divide(
        valid_counts,
        overall_class_images,
        out=np.zeros_like(valid_counts, dtype=float),
        where=overall_class_images > 0,
    )
    strat_error = float(np.mean(np.abs(valid_fraction - VAL_RATIO)))
    # train class 누락은 사실상 허용하지 않고, valid 누락도 강한 penalty를 줍니다.
    # 사전식 비교: train 누락 0개가 어떤 비율 개선보다 항상 우선합니다.
    score = (missing_train, missing_valid, ratio_error * 10 + strat_error)
    return score, train_stems_, valid_stems_, train_counts, valid_counts


best = None
for candidate_seed in range(SEED, SEED + N_SPLIT_CANDIDATES):
    splitter = MultilabelStratifiedShuffleSplit(
        n_splits=1,
        test_size=free_valid_ratio,
        random_state=candidate_seed,
    )
    free_train_local, free_valid_local = next(
        splitter.split(
            np.zeros((len(free_family_idx), 1)),
            family_labels[free_family_idx],
        )
    )
    train_idx = np.asarray(sorted(
        forced_train_family_idx | set(free_family_idx[free_train_local].tolist())
    ))
    valid_idx = free_family_idx[free_valid_local]
    candidate = split_score(train_idx, valid_idx)
    if best is None or candidate[0] < best[0]:
        best = (*candidate, candidate_seed, train_idx, valid_idx)

(
    best_score,
    train_stems,
    valid_stems,
    train_class_counts,
    valid_class_counts,
    selected_seed,
    train_family_idx,
    valid_family_idx,
) = best

train_stems = sorted(train_stems)
valid_stems = sorted(valid_stems)
train_families = {family_ids[i] for i in train_family_idx}
valid_families = {family_ids[i] for i in valid_family_idx}

assert not (train_families & valid_families), "family leakage가 발견되었습니다."
assert not (set(train_stems) & set(valid_stems))
assert len(train_stems) + len(valid_stems) == len(image_records)
if (train_class_counts == 0).any():
    raise RuntimeError("Train에 빠진 클래스가 있습니다. split 알고리즘을 점검하세요.")

print(f"선택 seed  : {selected_seed}")
print(f"Train      : {len(train_stems):,}장 / {len(train_families):,} families")
print(f"Valid      : {len(valid_stems):,}장 / {len(valid_families):,} families")
print(f"Valid 비율 : {len(valid_stems) / len(image_records):.2%}")
print(f"Valid 누락 클래스: {int((valid_class_counts == 0).sum())}")

if (valid_class_counts == 0).any():
    missing_ids = np.flatnonzero(valid_class_counts == 0).tolist()
    print("경고 - valid 누락 클래스:", [model_id_to_code[i] for i in missing_ids])

split_stats = mapping_preview.copy()
split_stats["all_images"] = overall_class_images
split_stats["train_images"] = train_class_counts
split_stats["valid_images"] = valid_class_counts
display(split_stats.sort_values("all_images").head(20))


## 7. RF-DETR용 COCO dataset 생성

생성 구조:

```text
rfdetr_xlarge_coco_grouped/
├─ train/
│  ├─ _annotations.coco.json
│  └─ images...
└─ valid/
   ├─ _annotations.coco.json
   └─ images...
```


In [ ]:
COCO_ROOT = DATA_ROOT / f"rfdetr_xlarge_coco_grouped_seed{selected_seed}_val{int(VAL_RATIO*100)}"
REBUILD_COCO_DATASET = True


def hardlink_or_copy(source: Path, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        return
    try:
        os.link(source, destination)
    except OSError:
        shutil.copy2(source, destination)


if REBUILD_COCO_DATASET and COCO_ROOT.exists():
    if not COCO_ROOT.name.startswith("rfdetr_xlarge_coco_grouped_"):
        raise RuntimeError(f"안전하지 않은 삭제 대상: {COCO_ROOT}")
    shutil.rmtree(COCO_ROOT)

categories = [
    {
        "id": model_id,
        "name": f"{model_id_to_code[model_id]} | {model_id_to_name[model_id]}",
        "supercategory": "none",
    }
    for model_id in range(num_classes)
]


def export_coco_split(split_name: str, stems: list[str]) -> Path:
    split_dir = COCO_ROOT / split_name
    split_dir.mkdir(parents=True, exist_ok=True)
    images = []
    annotations = []
    annotation_id = 1

    for image_id, stem in enumerate(tqdm(stems, desc=f"COCO {split_name}"), start=1):
        record = image_records[stem]
        source = Path(record["image_path"])
        destination = split_dir / record["file_name"]
        hardlink_or_copy(source, destination)

        images.append({
            "id": image_id,
            "file_name": record["file_name"],
            "width": record["width"],
            "height": record["height"],
        })

        for ann in record["annotations"]:
            x, y, bw, bh = ann["bbox"]
            annotations.append({
                "id": annotation_id,
                "image_id": image_id,
                "category_id": code_to_model_id[ann["category_code"]],
                "bbox": [x, y, bw, bh],
                "area": bw * bh,
                "iscrowd": 0,
            })
            annotation_id += 1

    coco = {
        "info": {
            "description": "AI Hub pill detection - group-safe split",
            "version": "1.0",
        },
        "licenses": [],
        "images": images,
        "categories": categories,
        "annotations": annotations,
    }
    annotation_file = split_dir / "_annotations.coco.json"
    annotation_file.write_text(
        json.dumps(coco, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    return annotation_file


train_coco_path = export_coco_split("train", train_stems)
valid_coco_path = export_coco_split("valid", valid_stems)

class_mapping = {
    "model": "RF-DETR-XL",
    "num_classes": num_classes,
    "classes": mapping_preview.to_dict(orient="records"),
}
(COCO_ROOT / "class_mapping.json").write_text(
    json.dumps(class_mapping, ensure_ascii=False, indent=2), encoding="utf-8"
)

split_manifest = {
    "seed": selected_seed,
    "val_ratio": VAL_RATIO,
    "group_method": "flip-invariant class+bbox geometry signature",
    "group_geometry_decimals": GROUP_GEOMETRY_DECIMALS,
    "train": train_stems,
    "valid": valid_stems,
    "family_to_stems": dict(family_to_stems),
}
(COCO_ROOT / "split_manifest.json").write_text(
    json.dumps(split_manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

print(f"COCO_ROOT: {COCO_ROOT}")
print(f"Train JSON: {train_coco_path}")
print(f"Valid JSON: {valid_coco_path}")


## 8. COCO 무결성 검사 · label 시각화


In [ ]:
for annotation_file, expected_images in [
    (train_coco_path, len(train_stems)),
    (valid_coco_path, len(valid_stems)),
]:
    coco_api = COCO(str(annotation_file))
    assert len(coco_api.imgs) == expected_images
    assert len(coco_api.cats) == num_classes
    assert set(coco_api.cats) == set(range(num_classes))
    assert all(ann["bbox"][2] > 0 and ann["bbox"][3] > 0 for ann in coco_api.anns.values())
    print(
        f"{annotation_file.parent.name:>5}: "
        f"images={len(coco_api.imgs):,}, annotations={len(coco_api.anns):,}, "
        f"categories={len(coco_api.cats)}"
    )


def show_ground_truth(stem: str, figsize=(10, 8)) -> None:
    record = image_records[stem]
    image = np.asarray(Image.open(record["image_path"]).convert("RGB")).copy()
    detections = sv.Detections(
        xyxy=np.asarray([
            [x, y, x + bw, y + bh]
            for x, y, bw, bh in (ann["bbox"] for ann in record["annotations"])
        ], dtype=float),
        class_id=np.asarray([
            code_to_model_id[ann["category_code"]] for ann in record["annotations"]
        ], dtype=int),
    )
    labels = [
        f"{model_id_to_code[int(class_id)]} {model_id_to_name[int(class_id)]}"
        for class_id in detections.class_id
    ]
    annotated = sv.BoxAnnotator(thickness=3).annotate(image.copy(), detections)
    annotated = sv.LabelAnnotator(text_scale=0.45).annotate(annotated, detections, labels)
    plt.figure(figsize=figsize)
    plt.imshow(annotated)
    plt.title(f"Ground truth · {stem}")
    plt.axis("off")
    plt.show()


show_ground_truth(random.choice(valid_stems))


## 9. W&B 로그인 · RF-DETR-XL 생성

`ACCEPT_PLATFORM_MODEL_LICENSE=True`는 사용자가 PML-1.0을 검토하고 동의했다는 뜻입니다.
모델 다운로드/사용 과정에서 Roboflow 계정 인증이 추가로 요구될 수 있습니다.


In [ ]:
if not ACCEPT_PLATFORM_MODEL_LICENSE:
    raise RuntimeError(
        "PML-1.0을 검토하고 동의했다면 설정 셀의 "
        "ACCEPT_PLATFORM_MODEL_LICENSE = True로 바꾼 뒤 다시 실행하세요."
    )

if USE_WANDB and (RUN_TRAINING or RESUME_TRAINING):
    os.environ["WANDB_ENTITY"] = WANDB_ENTITY
    wandb.login()

model = None
if RUN_TRAINING:
    model = RFDETRXLarge(
        accept_platform_model_license=True,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
    )
    print("RF-DETR-XL pretrained model 준비 완료")
elif RESUME_TRAINING:
    print("재개 학습 모드: 다음 재개 셀에서 모델을 생성합니다.")
else:
    print("학습 없이 기존 best checkpoint를 평가합니다.")


## 10. 학습

- AdamW, base LR `1e-4`, encoder LR `1.5e-4`
- cosine scheduler, warmup 1 epoch
- EMA 사용
- 첫 baseline은 multi-scale/crop jitter를 끄고 고정 560으로 학습
- 물체가 3~4개인 데이터라 validation의 max detections를 20으로 제한
- `last.ckpt`는 완전 재개용, `checkpoint_best_total.pth`는 best 추론용입니다.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
existing_checkpoints = sorted(OUTPUT_DIR.glob("*.ckpt")) + sorted(OUTPUT_DIR.glob("*.pth"))
if RUN_TRAINING and existing_checkpoints:
    raise RuntimeError(
        f"기존 checkpoint가 있습니다: {existing_checkpoints[:3]}\n"
        "이어 학습하려면 RUN_TRAINING=False, RESUME_TRAINING=True로 바꾸고, "
        "새 실험이면 RUN_NAME을 바꾸세요."
    )

train_kwargs = dict(
    dataset_dir=str(COCO_ROOT),
    output_dir=str(OUTPUT_DIR),
    epochs=EPOCHS,
    resolution=RESOLUTION,
    batch_size=TRAIN_BATCH,
    eval_batch_size=EVAL_BATCH_SIZE,
    auto_batch_target_effective=AUTO_BATCH_EFFECTIVE,
    lr=LR,
    lr_encoder=LR_ENCODER,
    optimizer="adamw",
    weight_decay=WEIGHT_DECAY,
    lr_scheduler="cosine",
    lr_scheduler_kwargs={"min_factor": 0.01},
    warmup_epochs=1.0,
    use_ema=True,
    eval_base_model=False,
    checkpoint_interval=5,
    skip_best_epochs=1,
    early_stopping=EARLY_STOPPING,
    early_stopping_patience=8,
    early_stopping_min_delta=0.001,
    early_stopping_use_ema=True,
    best_model_metric="map",
    eval_max_dets=EVAL_MAX_DETECTIONS,
    eval_interval=1,
    log_per_class_metrics=False,
    compute_val_loss="auto",
    multi_scale=False,
    expanded_scales=False,
    scale_jitter=False,
    device="cuda",
    seed=SEED,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
    tensorboard=True,
    wandb=USE_WANDB,
    project=WANDB_PROJECT,
    run=RUN_NAME,
    progress_bar="tqdm",
)

display(pd.Series(train_kwargs, name="value").to_frame())
if RUN_TRAINING:
    model.train(**train_kwargs)
else:
    print("RUN_TRAINING=False: 학습 셀을 건너뜁니다.")

if RUN_TRAINING and USE_WANDB and wandb.run is not None:
    wandb.finish()


## 11. 중단된 학습 이어하기 (필요할 때만 실행)

같은 `output_dir`의 `last.ckpt`를 사용해야 optimizer/scheduler/best-score 상태까지 이어집니다.
`epochs`는 **총 목표 epoch**입니다.


In [ ]:
LAST_CHECKPOINT = OUTPUT_DIR / "last.ckpt"

if RESUME_TRAINING:
    if not LAST_CHECKPOINT.exists():
        raise FileNotFoundError(LAST_CHECKPOINT)

    resume_model = RFDETRXLarge(
        accept_platform_model_license=True,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
    )
    resume_kwargs = dict(train_kwargs)
    resume_kwargs["resume"] = str(LAST_CHECKPOINT)
    resume_model.train(**resume_kwargs)
    if USE_WANDB and wandb.run is not None:
        wandb.finish()
else:
    print("RESUME_TRAINING=False: 재개 학습 셀을 건너뜁니다.")


## 12. best checkpoint 로드 · RF-DETR 기본 COCO 평가


In [ ]:
BEST_CHECKPOINT = OUTPUT_DIR / "checkpoint_best_total.pth"
if not BEST_CHECKPOINT.exists():
    fallback = OUTPUT_DIR / "last_ema.pth"
    if fallback.exists():
        BEST_CHECKPOINT = fallback
    else:
        raise FileNotFoundError(
            f"best checkpoint를 찾지 못했습니다: {OUTPUT_DIR.resolve()}"
        )

# XL 모델 두 개가 GPU에 동시에 올라가는 일을 막습니다.
if "model" in globals() and model is not None:
    del model
if "resume_model" in globals():
    del resume_model
gc.collect()
torch.cuda.empty_cache()

best_model = RFDETRXLarge.from_checkpoint(
    str(BEST_CHECKPOINT),
    accept_platform_model_license=True,
)
print(f"Loaded: {BEST_CHECKPOINT}")

expected_checkpoint_names = [
    f"{model_id_to_code[i]} | {model_id_to_name[i]}" for i in range(num_classes)
]
actual_checkpoint_names = list(best_model.class_names)
if actual_checkpoint_names != expected_checkpoint_names:
    raise RuntimeError(
        "Checkpoint class 순서가 class_mapping.json과 다릅니다. "
        f"expected[:3]={expected_checkpoint_names[:3]}, "
        f"actual[:3]={actual_checkpoint_names[:3]}"
    )

official_eval = best_model.evaluate(
    dataset_dir=str(COCO_ROOT),
    split="val",
    resolution=RESOLUTION,
    batch_size=EVAL_BATCH_SIZE,
    eval_max_dets=EVAL_MAX_DETECTIONS,
    device="cuda",
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=False,
)
print(official_eval)


## 13. 대회 지표 mAP@[0.75:0.95] 직접 계산 (base) · 추가 기능 공통 유틸

RF-DETR의 기본 COCO mAP@[0.50:0.95]와 대회 지표는 다릅니다. `map20`은 이미지당 최대 20개, `map8`은 실제 제출과 같은 최대 8개 조건입니다.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 추가 기능 공통 유틸 · 검출 어댑터 · 평가 함수 (v2)
# ══════════════════════════════════════════════════════════════
import contextlib

RUN_OUTPUT_DIR = Path(RUN_OUTPUT_DIR) if "RUN_OUTPUT_DIR" in globals() else Path(OUTPUT_DIR)
ADDON_DIR = RUN_OUTPUT_DIR / "addons"
ADDON_DIR.mkdir(parents=True, exist_ok=True)


def batched(items, batch_size):
    for start in range(0, len(items), batch_size):
        yield items[start:start + batch_size]


def make_pred(xyxy, scores, labels, wh, max_det=None):
    """검출 결과를 점수 내림차순 dict로 통일: xyxy(N,4) float32, scores(N,), labels(N,), wh=(W,H)."""
    xyxy = np.asarray(xyxy, dtype=np.float32).reshape(-1, 4)
    scores = np.asarray(scores, dtype=np.float32).reshape(-1)
    labels = np.asarray(labels).astype(np.int64).reshape(-1)
    order = np.argsort(-scores, kind="stable")
    if max_det is not None:
        order = order[:max_det]
    return {
        "xyxy": xyxy[order],
        "scores": scores[order],
        "labels": labels[order],
        "wh": (int(wh[0]), int(wh[1])),
    }


def clip_xyxy(xyxy, wh):
    out = np.asarray(xyxy, dtype=np.float32).reshape(-1, 4).copy()
    out[:, [0, 2]] = np.clip(out[:, [0, 2]], 0, wh[0])
    out[:, [1, 3]] = np.clip(out[:, [1, 3]], 0, wh[1])
    return out


def box_iou_pairwise(a, b):
    """같은 행끼리의 IoU. a, b: (N,4) xyxy."""
    a = np.asarray(a, dtype=np.float32).reshape(-1, 4)
    b = np.asarray(b, dtype=np.float32).reshape(-1, 4)
    ix1, iy1 = np.maximum(a[:, 0], b[:, 0]), np.maximum(a[:, 1], b[:, 1])
    ix2, iy2 = np.minimum(a[:, 2], b[:, 2]), np.minimum(a[:, 3], b[:, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    area_a = np.clip(a[:, 2] - a[:, 0], 0, None) * np.clip(a[:, 3] - a[:, 1], 0, None)
    area_b = np.clip(b[:, 2] - b[:, 0], 0, None) * np.clip(b[:, 3] - b[:, 1], 0, None)
    return inter / np.maximum(area_a + area_b - inter, 1e-6)


def detect_batch(paths, max_det=None, conf=None):
    """[이미지 경로] → [pred dict]. (RF-DETR 어댑터)"""
    max_det = EVAL_MAX_DETECTIONS if max_det is None else max_det
    conf = PRED_CONFIDENCE if conf is None else conf
    images = [Image.open(p).convert("RGB") for p in paths]
    try:
        detections = best_model.predict(images, threshold=conf)
        if isinstance(detections, sv.Detections):
            detections = [detections]
        if len(detections) != len(images):
            raise RuntimeError(f"예측 batch 길이 불일치: inputs={len(images)}, outputs={len(detections)}")
        out = []
        for image, det in zip(images, detections):
            n = len(det)
            xyxy = det.xyxy if n else np.zeros((0, 4))
            scores = det.confidence if (n and det.confidence is not None) else np.zeros(n)
            labels = det.class_id if (n and det.class_id is not None) else np.zeros(n, dtype=np.int64)
            out.append(make_pred(xyxy, scores, labels, image.size, max_det))
    finally:
        for image in images:
            image.close()
    return out


def build_targets(stems):
    targets = {}
    for stem in stems:
        anns = image_records[stem]["annotations"]
        boxes = np.array(
            [[x, y, x + w, y + h] for x, y, w, h in (a["bbox"] for a in anns)],
            dtype=np.float32,
        ).reshape(-1, 4)
        labels = np.array([code_to_model_id[a["category_code"]] for a in anns], dtype=np.int64)
        targets[stem] = {"boxes": torch.as_tensor(boxes), "labels": torch.as_tensor(labels)}
    return targets


def _metric_dict(pred, n):
    return {
        "boxes": torch.as_tensor(pred["xyxy"][:n], dtype=torch.float32).reshape(-1, 4),
        "scores": torch.as_tensor(pred["scores"][:n], dtype=torch.float32),
        "labels": torch.as_tensor(pred["labels"][:n], dtype=torch.int64),
    }


def evaluate_variant(pred_map, stems, targets, chunk=64):
    """대회 지표 mAP@[.75:.95]. map20=이미지당 최대 20개, map8=제출과 같은 최대 8개."""
    metric20 = MeanAveragePrecision(
        box_format="xyxy", iou_type="bbox", iou_thresholds=COMPETITION_IOUS,
        max_detection_thresholds=[1, 10, EVAL_MAX_DETECTIONS],
        class_metrics=False, backend="faster_coco_eval",
    )
    metric8 = MeanAveragePrecision(
        box_format="xyxy", iou_type="bbox", iou_thresholds=COMPETITION_IOUS,
        max_detection_thresholds=[1, 5, SUBMISSION_MAX_DETECTIONS],
        class_metrics=False, backend="faster_coco_eval",
    )
    for stem_chunk in batched(list(stems), chunk):
        preds20 = [_metric_dict(pred_map[s], EVAL_MAX_DETECTIONS) for s in stem_chunk]
        preds8 = [_metric_dict(pred_map[s], SUBMISSION_MAX_DETECTIONS) for s in stem_chunk]
        tgts = [targets[s] for s in stem_chunk]
        metric20.update(preds20, tgts)
        metric8.update(preds8, tgts)
    return {
        "map20": float(metric20.compute()["map"]),
        "map8": float(metric8.compute()["map"]),
    }


In [ ]:
# ══ 검증셋 검출 캐시 + 기본(base) 대회 지표 ══
# 검출기를 val에 한 번만 돌려 두고, SAM / SimCLR 변형은 이 캐시에서 계산합니다.
val_paths = {stem: Path(image_records[stem]["image_path"]) for stem in valid_stems}
val_targets = build_targets(valid_stems)

val_raw = {}
for stem_batch in tqdm(list(batched(valid_stems, PRED_BATCH_SIZE)), desc="Validation 검출"):
    batch_preds = detect_batch([val_paths[s] for s in stem_batch], max_det=EVAL_MAX_DETECTIONS)
    for stem, pred in zip(stem_batch, batch_preds):
        val_raw[stem] = pred

variant_scores = {"base": evaluate_variant(val_raw, valid_stems, val_targets)}
competition_map = variant_scores["base"]["map20"]
submission_cap_map = variant_scores["base"]["map8"]
print(f"[base] Competition mAP@[.75:.95] (max {EVAL_MAX_DETECTIONS}): {competition_map:.5f}")
print(f"[base] Submission-cap mAP@[.75:.95] (max {SUBMISSION_MAX_DETECTIONS}): {submission_cap_map:.5f}")


## 14. (추가 기능 1) SAM 박스 정제 — `USE_SAM`

검출 박스를 SAM의 box prompt로 넣어 얻은 마스크의 외접 박스로 좌표를 바꿉니다. 학습이 없고, val에서 padding(px)을 자동 튜닝합니다.


In [ ]:
# ══ SAM: 검출 박스 → 마스크 외접 박스로 정제 (USE_SAM) ══
# 검출 박스를 SAM의 box prompt로 넣어 마스크를 얻고, 마스크의 외접 박스로 좌표를 바꿉니다.
# 마스크가 이상하면(원본과 IoU 낮음 / 면적 너무 작음) 해당 박스는 원본을 그대로 유지합니다.
SAM_ACTIVE = False
sam_model = None
val_sam = None                       # stem -> (N,4) SAM 마스크 박스(정제 안 한 행은 NaN)
SAM_DEVICE = 0 if torch.cuda.is_available() else "cpu"
_sam_fail_count = 0


def _largest_component_box(mask_bool):
    """가장 큰 연결 성분의 외접 박스와 면적. 없으면 None."""
    n_labels, _, stats, _ = cv2.connectedComponentsWithStats(
        mask_bool.astype(np.uint8), connectivity=8
    )
    if n_labels <= 1:
        return None
    idx = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    x, y, w, h, area = [int(v) for v in stats[idx]]
    return [float(x), float(y), float(x + w), float(y + h)], area


def sam_mask_boxes(image_path, boxes_xyxy, wh):
    """boxes (M,4) → (M,4) 마스크 외접 박스. 실패한 행은 NaN."""
    boxes = np.asarray(boxes_xyxy, dtype=np.float32).reshape(-1, 4)
    out = np.full((len(boxes), 4), np.nan, dtype=np.float32)
    if len(boxes) == 0:
        return out
    results = sam_model.predict(
        str(image_path), bboxes=boxes.tolist(), device=SAM_DEVICE, verbose=False
    )
    if not results or results[0].masks is None:
        return out
    masks = results[0].masks.data
    masks = masks.detach().cpu().numpy() if hasattr(masks, "detach") else np.asarray(masks)
    if masks.ndim == 2:
        masks = masks[None]
    if len(masks) != len(boxes):
        return out
    for i, mask in enumerate(masks):
        found = _largest_component_box(mask > 0.5)
        if found is None:
            continue
        box, area = found
        mask_h, mask_w = mask.shape[:2]
        sx, sy = wh[0] / mask_w, wh[1] / mask_h       # 마스크가 원본 해상도가 아닐 때 보정
        box = [box[0] * sx, box[1] * sy, box[2] * sx, box[3] * sy]
        box_area = max(float((boxes[i, 2] - boxes[i, 0]) * (boxes[i, 3] - boxes[i, 1])), 1.0)
        if area * sx * sy < SAM_MIN_AREA_RATIO * box_area:
            continue
        out[i] = box
    return out


def sam_boxes_for_pred(image_path, pred):
    """pred의 상위 SAM_TOPK개(점수 ≥ SAM_MIN_SCORE)에 대한 마스크 박스. 나머지는 NaN."""
    global _sam_fail_count
    n = len(pred["scores"])
    out = np.full((n, 4), np.nan, dtype=np.float32)
    idx = [i for i in range(min(n, SAM_TOPK)) if pred["scores"][i] >= SAM_MIN_SCORE]
    if not idx:
        return out
    boxes = clip_xyxy(pred["xyxy"][idx], pred["wh"])
    keep = (boxes[:, 2] - boxes[:, 0] > 1) & (boxes[:, 3] - boxes[:, 1] > 1)
    idx = [i for i, k in zip(idx, keep) if k]
    boxes = boxes[keep]
    if not idx:
        return out
    try:
        out[idx] = sam_mask_boxes(image_path, boxes, pred["wh"])
    except Exception as exc:                        # 한 장 실패가 전체를 멈추지 않게 함
        _sam_fail_count += 1
        if _sam_fail_count <= 3:
            print(f"⚠ SAM 추론 실패(원본 박스 유지): {type(exc).__name__}: {str(exc)[:150]}")
    return out


def apply_sam(pred, sam_xyxy, pad=None, blend=None, min_iou=None):
    """pred의 박스를 SAM 박스로 교체(조건 만족 행만). 새 pred dict를 반환."""
    pad = SAM_PAD_PX if pad is None else pad
    blend = SAM_BLEND if blend is None else blend
    min_iou = SAM_MIN_IOU if min_iou is None else min_iou
    new = pred["xyxy"].copy()
    valid = ~np.isnan(sam_xyxy).any(axis=1)
    if valid.any():
        sam_box = sam_xyxy[valid]
        orig = pred["xyxy"][valid]
        ref = sam_box + np.array([-pad, -pad, pad, pad], dtype=np.float32)
        mixed = clip_xyxy(blend * ref + (1.0 - blend) * orig, pred["wh"])
        good = (
            (box_iou_pairwise(sam_box, orig) >= min_iou)
            & (mixed[:, 2] - mixed[:, 0] > 1.0)
            & (mixed[:, 3] - mixed[:, 1] > 1.0)
        )
        rows = np.flatnonzero(valid)[good]
        new[rows] = mixed[good]
    out = dict(pred)
    out["xyxy"] = new
    return out


def setup_sam():
    global sam_model
    from ultralytics import SAM
    last_error = None
    for name in SAM_MODEL_CANDIDATES:
        try:
            sam_model = SAM(name)
            # 자체 점검: GT 박스를 prompt로 넣었을 때 마스크가 나오는지, GT 박스 스타일(타이트한지) 확인
            ious = []
            for stem in valid_stems[:20]:
                rec = image_records[stem]
                gt = np.array([[x, y, x + w, y + h] for x, y, w, h in (a["bbox"] for a in rec["annotations"])],
                              dtype=np.float32)
                got = sam_mask_boxes(val_paths[stem], gt, (rec["width"], rec["height"]))
                ok = ~np.isnan(got).any(axis=1)
                if ok.any():
                    ious.extend(box_iou_pairwise(got[ok], gt[ok]).tolist())
            if not ious:
                raise RuntimeError("자체 점검 실패: 마스크가 반환되지 않았습니다.")
            print(f"SAM 로드: {name} | GT박스 vs SAM마스크박스 평균 IoU = {np.mean(ious):.3f} "
                  f"(1.0에 가까울수록 GT 박스가 마스크에 타이트)")
            return name
        except Exception as exc:
            last_error = f"{name}: {type(exc).__name__}: {str(exc)[:150]}"
            print(f"  SAM 후보 실패 → {last_error}")
            sam_model = None
    raise RuntimeError(f"SAM 후보를 모두 로드하지 못했습니다. 마지막 오류: {last_error}")


if USE_SAM:
    try:
        setup_sam()
        val_sam = {}
        for stem in tqdm(valid_stems, desc="SAM 정제 (val)"):
            val_sam[stem] = sam_boxes_for_pred(val_paths[stem], val_raw[stem])
        n_refined = sum(int((~np.isnan(v).any(axis=1)).sum()) for v in val_sam.values())
        n_total = sum(min(len(v), SAM_TOPK) for v in val_sam.values())
        print(f"SAM 마스크 박스 확보: {n_refined:,} / {n_total:,}")
        if n_refined == 0:
            raise RuntimeError("SAM 마스크 박스를 하나도 얻지 못했습니다.")
        SAM_ACTIVE = True
    except Exception:
        import traceback
        traceback.print_exc()
        print("⚠ SAM 준비에 실패해 SAM 없이 진행합니다. (위 오류를 확인하세요)")
        SAM_ACTIVE = False
else:
    print("USE_SAM=False: SAM 단계를 건너뜁니다.")

if SAM_ACTIVE:
    def _sam_map(pad):
        return {s: apply_sam(val_raw[s], val_sam[s], pad=pad) for s in valid_stems}

    if SAM_TUNE_PAD:
        pad_rows = []
        for pad in SAM_PAD_GRID:
            pad_rows.append({"pad_px": float(pad), **evaluate_variant(_sam_map(pad), valid_stems, val_targets)})
        pad_table = pd.DataFrame(pad_rows)
        pad_table["abs_pad"] = pad_table["pad_px"].abs()
        display(pad_table.drop(columns="abs_pad"))
        best_row = pad_table.sort_values(["map8", "map20", "abs_pad"], ascending=[False, False, True]).iloc[0]
        SAM_PAD_PX = float(best_row["pad_px"])
        print(f"SAM_PAD_PX 자동 선택: {SAM_PAD_PX:+.1f}px (val 기준, 단일 스칼라 튜닝)")
    variant_scores["sam"] = evaluate_variant(_sam_map(SAM_PAD_PX), valid_stems, val_targets)
    print(f"[sam] mAP@[.75:.95] max20={variant_scores['sam']['map20']:.5f} | "
          f"max8={variant_scores['sam']['map8']:.5f}")

    def show_sam_compare(stem, top=4):
        import matplotlib.patches as patches
        pred = val_raw[stem]
        refined = apply_sam(pred, val_sam[stem])
        image = Image.open(val_paths[stem]).convert("RGB")
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.imshow(image)
        for i in range(min(top, len(pred["scores"]))):
            for box, color, style in ((pred["xyxy"][i], "red", "--"), (refined["xyxy"][i], "lime", "-")):
                ax.add_patch(patches.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1],
                                               fill=False, edgecolor=color, linewidth=1.5, linestyle=style))
        ax.set_title(f"{stem}  (빨강 점선=원본, 초록=SAM 정제)")
        ax.axis("off")
        plt.show()
        image.close()

    try:
        for sample_stem in random.sample(valid_stems, min(2, len(valid_stems))):
            show_sam_compare(sample_stem)
    except Exception as exc:
        print("SAM 비교 시각화 생략:", type(exc).__name__, exc)


## 15. (추가 기능 2) SimCLR crop 분류기 — `USE_SIMCLR`

train GT crop으로 대조학습(SimCLR) → 118종 분류 fine-tuning 후, 검출 박스를 crop해 클래스를 재판정하고 검출기 클래스와 융합합니다.
RF-DETR-XL과 같은 GPU를 쓰므로 실행 전에 VRAM을 정리하며, OOM이면 SimCLR 배치를 자동으로 줄입니다.


In [ ]:
# ══ SimCLR crop 분류기 (USE_SIMCLR) ══
# 1) train split의 GT crop만으로 대조학습(SimCLR) → 2) 118종 분류 fine-tuning
# 3) 검출된 박스를 crop해 클래스 확률을 얻고, 검출기 클래스와 융합합니다.
# val/test 이미지는 학습에 쓰지 않습니다(누수 방지). GPU 위에서 증강해 DataLoader 없이 동작합니다.
import torch.nn as nn
import torch.nn.functional as F
import torchvision

SIMCLR_ACTIVE = False
simclr_net = None
SIMCLR_FUSE_ALPHA_FINAL = SIMCLR_FUSE_ALPHA
SIMCLR_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
_IMNET_MEAN = (0.485, 0.456, 0.406)
_IMNET_STD = (0.229, 0.224, 0.225)
if SIMCLR_DEVICE == "cuda" and torch.cuda.is_bf16_supported():
    SIMCLR_AMP_DTYPE = torch.bfloat16
elif SIMCLR_DEVICE == "cuda":
    SIMCLR_AMP_DTYPE = torch.float16
else:
    SIMCLR_AMP_DTYPE = None


def amp_ctx():
    if SIMCLR_AMP_DTYPE is None:
        return contextlib.nullcontext()
    return torch.autocast(device_type="cuda", dtype=SIMCLR_AMP_DTYPE)


def make_scaler():
    if SIMCLR_AMP_DTYPE == torch.float16:
        return torch.amp.GradScaler("cuda")
    return None


def backward_step(loss, optimizer, scaler, params, clip=5.0):
    optimizer.zero_grad(set_to_none=True)
    if scaler is not None:
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(params, clip)
        scaler.step(optimizer)
        scaler.update()
    else:
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, clip)
        optimizer.step()


# ── 모델 ─────────────────────────────────────────────────────
class PillNet(nn.Module):
    def __init__(self, backbone="resnet50", imagenet_init=True, proj_dim=128, num_classes=0):
        super().__init__()
        if not str(backbone).startswith("resnet"):
            raise ValueError("SIMCLR_BACKBONE은 resnet18 / resnet34 / resnet50 중에서 고르세요.")
        weights = None
        if imagenet_init:
            try:
                weights = torchvision.models.get_model_weights(backbone).DEFAULT
            except Exception as exc:
                print("ImageNet 가중치 정보를 찾지 못해 랜덤 초기화:", exc)
        try:
            net = torchvision.models.get_model(backbone, weights=weights)
        except Exception as exc:
            print(f"ImageNet 가중치를 받지 못해 랜덤 초기화로 진행: {type(exc).__name__}")
            net = torchvision.models.get_model(backbone, weights=None)
        self.feat_dim = int(net.fc.in_features)
        net.fc = nn.Identity()
        self.backbone = net
        self.proj = nn.Sequential(
            nn.Linear(self.feat_dim, 512), nn.ReLU(inplace=True), nn.Linear(512, proj_dim)
        )
        self.head = nn.Linear(self.feat_dim, num_classes) if num_classes > 0 else None
        self.register_buffer("mean", torch.tensor(_IMNET_MEAN).view(1, 3, 1, 1), persistent=False)
        self.register_buffer("std", torch.tensor(_IMNET_STD).view(1, 3, 1, 1), persistent=False)

    def features(self, x01):
        return self.backbone((x01 - self.mean) / self.std)

    def embed(self, x01):
        return F.normalize(self.proj(self.features(x01)).float(), dim=1)

    def classify(self, x01):
        return self.head(self.features(x01))


# ── crop · GPU 증강 ──────────────────────────────────────────
def crop_square(img, box, size, pad_ratio):
    """박스 중심 기준 정사각 crop(여백 포함, 이미지 밖은 검정) → size×size."""
    x1, y1, x2, y2 = [float(v) for v in box]
    cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
    side = max(x2 - x1, y2 - y1, 8.0) * (1.0 + pad_ratio)
    s = max(int(round(side)), 2)
    left, top = int(round(cx - s / 2.0)), int(round(cy - s / 2.0))
    return img.crop((left, top, left + s, top + s)).resize((size, size), Image.BILINEAR)


def collect_gt_crops(stems, desc):
    total = sum(len(image_records[s]["annotations"]) for s in stems)
    size = SIMCLR_CROP
    crops = np.empty((total, size, size, 3), dtype=np.uint8)
    labels = np.empty(total, dtype=np.int64)
    k = 0
    for stem in tqdm(stems, desc=desc):
        record = image_records[stem]
        with Image.open(record["image_path"]) as opened:
            image = opened.convert("RGB")
        for ann in record["annotations"]:
            x, y, w, h = ann["bbox"]
            crops[k] = np.asarray(crop_square(image, (x, y, x + w, y + h), size, SIMCLR_CROP_PAD))
            labels[k] = code_to_model_id[ann["category_code"]]
            k += 1
        image.close()
    return crops, labels


def gpu_augment(x, out_size, scale=(0.4, 1.0), color=0.3, noise=0.02):
    """x: (B,3,H,W) float[0,1] → 랜덤 확대·이동·회전·반전 + 약한 색 변형. 알약은 색이 중요해 hue는 건드리지 않음."""
    batch, channels = x.shape[0], x.shape[1]
    dev = x.device
    zoom = torch.empty(batch, device=dev).uniform_(scale[0], scale[1]).sqrt()
    angle = torch.rand(batch, device=dev) * (2.0 * math.pi)
    flip = (torch.rand(batch, device=dev) < 0.5).float() * 2.0 - 1.0
    max_shift = 1.0 - zoom
    tx = (torch.rand(batch, device=dev) * 2.0 - 1.0) * max_shift
    ty = (torch.rand(batch, device=dev) * 2.0 - 1.0) * max_shift
    cos, sin = torch.cos(angle), torch.sin(angle)
    theta = torch.zeros(batch, 2, 3, device=dev)
    theta[:, 0, 0] = zoom * cos * flip
    theta[:, 0, 1] = -zoom * sin
    theta[:, 0, 2] = tx
    theta[:, 1, 0] = zoom * sin * flip
    theta[:, 1, 1] = zoom * cos
    theta[:, 1, 2] = ty
    grid = F.affine_grid(theta, (batch, channels, out_size, out_size), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="reflection", align_corners=False)
    if color > 0:
        bright = 1.0 + (torch.rand(batch, 1, 1, 1, device=dev) * 2.0 - 1.0) * color
        contrast = 1.0 + (torch.rand(batch, 1, 1, 1, device=dev) * 2.0 - 1.0) * color
        satur = 1.0 + (torch.rand(batch, 1, 1, 1, device=dev) * 2.0 - 1.0) * color
        x = x * bright
        mean = x.mean(dim=(1, 2, 3), keepdim=True)
        x = (x - mean) * contrast + mean
        gray = 0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]
        x = (x - gray) * satur + gray
    if noise > 0:
        x = x + torch.randn_like(x) * noise
    return x.clamp(0.0, 1.0)


def nt_xent(z, temperature):
    n = z.shape[0] // 2
    sim = (z @ z.t()) / temperature
    sim = sim.masked_fill(torch.eye(2 * n, dtype=torch.bool, device=z.device), float("-inf"))
    targets = torch.cat([torch.arange(n, 2 * n), torch.arange(0, n)]).to(z.device)
    return F.cross_entropy(sim, targets)


def _cosine_with_warmup(warm_steps, total_steps, floor=0.01):
    def fn(step):
        warm = min(1.0, (step + 1) / max(1, warm_steps))
        progress = min(1.0, step / max(1, total_steps))
        return warm * (floor + (1.0 - floor) * 0.5 * (1.0 + math.cos(math.pi * progress)))
    return fn


# ── 학습 ─────────────────────────────────────────────────────
def pretrain_simclr(crops_np, batch):
    net = PillNet(SIMCLR_BACKBONE, SIMCLR_IMAGENET_INIT, num_classes=0).to(SIMCLR_DEVICE)
    if SIMCLR_EPOCHS <= 0:
        return net
    crops = torch.from_numpy(crops_np)                      # NHWC uint8 (메모리 공유)
    n = len(crops)
    batch = max(2, min(batch, n))
    steps_per_epoch = max(1, n // batch)
    total_steps = SIMCLR_EPOCHS * steps_per_epoch
    params = list(net.backbone.parameters()) + list(net.proj.parameters())
    optimizer = torch.optim.AdamW(params, lr=SIMCLR_LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, _cosine_with_warmup(steps_per_epoch, total_steps)
    )
    scaler = make_scaler()
    net.train()
    for epoch in range(SIMCLR_EPOCHS):
        perm = torch.randperm(n)
        running, count = 0.0, 0
        for it in range(steps_per_epoch):
            idx = perm[it * batch:(it + 1) * batch]
            xb = crops[idx].to(SIMCLR_DEVICE, non_blocking=True).permute(0, 3, 1, 2).float().div_(255.0)
            with torch.no_grad():
                v1 = gpu_augment(xb, SIMCLR_CROP, scale=(0.35, 1.0), color=0.30)
                v2 = gpu_augment(xb, SIMCLR_CROP, scale=(0.35, 1.0), color=0.30)
            with amp_ctx():
                z = net.embed(torch.cat([v1, v2], dim=0))
            loss = nt_xent(z, SIMCLR_TEMP)
            if not torch.isfinite(loss):
                optimizer.zero_grad(set_to_none=True)
                continue
            backward_step(loss, optimizer, scaler, params)
            scheduler.step()
            running += float(loss.detach())
            count += 1
        print(f"[SimCLR 사전학습] epoch {epoch + 1:02d}/{SIMCLR_EPOCHS} | loss {running / max(1, count):.4f}")
    return net


@torch.no_grad()
def predict_probs(net, crops_nhwc, batch=256, tta=True):
    """crops (N,S,S,3) uint8 → (N, num_classes) softmax 확률(numpy)."""
    net.eval()
    outputs = []
    for start in range(0, len(crops_nhwc), batch):
        xb = torch.as_tensor(np.ascontiguousarray(crops_nhwc[start:start + batch]))
        xb = xb.to(SIMCLR_DEVICE).permute(0, 3, 1, 2).float().div_(255.0)
        views = [xb] + ([torch.rot90(xb, k, (2, 3)) for k in (1, 2, 3)] if tta else [])
        acc = 0.0
        for view in views:
            with amp_ctx():
                logits = net.classify(view)
            acc = acc + F.softmax(logits.float(), dim=1)
        outputs.append((acc / len(views)).cpu())
    if not outputs:
        return np.zeros((0, num_classes), dtype=np.float32)
    return torch.cat(outputs).numpy().astype(np.float32)


def finetune_classifier(net, tr_x, tr_y, va_x, va_y, batch):
    net.head = nn.Linear(net.feat_dim, num_classes).to(SIMCLR_DEVICE)
    crops = torch.from_numpy(tr_x)
    labels = torch.from_numpy(tr_y)
    n = len(crops)
    batch = max(2, min(batch, n))
    steps_per_epoch = max(1, n // batch)
    total_steps = SIMCLR_CLS_EPOCHS * steps_per_epoch
    groups = [
        {"params": list(net.backbone.parameters()), "lr": SIMCLR_CLS_LR_BACKBONE},
        {"params": list(net.head.parameters()), "lr": SIMCLR_CLS_LR_HEAD},
    ]
    all_params = groups[0]["params"] + groups[1]["params"]
    optimizer = torch.optim.AdamW(groups, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, _cosine_with_warmup(steps_per_epoch, total_steps)
    )
    scaler = make_scaler()
    for epoch in range(SIMCLR_CLS_EPOCHS):
        net.train()
        perm = torch.randperm(n)
        running, count = 0.0, 0
        for it in range(steps_per_epoch):
            idx = perm[it * batch:(it + 1) * batch]
            xb = crops[idx].to(SIMCLR_DEVICE, non_blocking=True).permute(0, 3, 1, 2).float().div_(255.0)
            yb = labels[idx].to(SIMCLR_DEVICE)
            with torch.no_grad():
                xb = gpu_augment(xb, SIMCLR_CROP, scale=(0.6, 1.0), color=0.12, noise=0.01)
            with amp_ctx():
                logits = net.classify(xb)
            loss = F.cross_entropy(logits.float(), yb, label_smoothing=0.1)
            if not torch.isfinite(loss):
                optimizer.zero_grad(set_to_none=True)
                continue
            backward_step(loss, optimizer, scaler, all_params)
            scheduler.step()
            running += float(loss.detach())
            count += 1
        message = f"[SimCLR 분류 fine-tune] epoch {epoch + 1:02d}/{SIMCLR_CLS_EPOCHS} | loss {running / max(1, count):.4f}"
        if va_x is not None and len(va_x) and ((epoch + 1) % 4 == 0 or epoch + 1 == SIMCLR_CLS_EPOCHS):
            acc = float((predict_probs(net, va_x, tta=False).argmax(1) == va_y).mean())
            message += f" | val GT-crop acc {acc:.4f}"
        print(message)
    return net


def train_simclr_classifier(tr_x, tr_y, va_x, va_y):
    batch = SIMCLR_BATCH
    while True:
        try:
            net = pretrain_simclr(tr_x, batch)
            net = finetune_classifier(net, tr_x, tr_y, va_x, va_y, batch)
            return net
        except torch.cuda.OutOfMemoryError:
            gc.collect()
            torch.cuda.empty_cache()
            if batch <= 16:
                raise
            batch //= 2
            print(f"⚠ GPU 메모리 부족 → SIMCLR 배치를 {batch}로 줄여 처음부터 다시 시도합니다.")


def setup_simclr():
    global simclr_net
    torch.manual_seed(SEED)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    train_hash = hashlib.sha1(",".join(train_stems).encode("utf-8")).hexdigest()[:10]
    ckpt_path = ADDON_DIR / (
        f"simclr_{SIMCLR_BACKBONE}_s{SIMCLR_CROP}_e{SIMCLR_EPOCHS}_c{SIMCLR_CLS_EPOCHS}_{train_hash}.pt"
    )
    net = None
    if SIMCLR_REUSE_CACHE and ckpt_path.exists():
        try:
            state = torch.load(ckpt_path, map_location="cpu")
            if state["sorted_codes"] != sorted_codes:
                raise ValueError("클래스 구성이 달라 캐시를 사용할 수 없습니다.")
            net = PillNet(SIMCLR_BACKBONE, False, num_classes=num_classes)
            net.load_state_dict(state["state_dict"])
            net.to(SIMCLR_DEVICE).eval()
            print(f"SimCLR 분류기 캐시 로드: {ckpt_path.name}")
        except Exception as exc:
            print(f"캐시 로드 실패 → 새로 학습합니다: {type(exc).__name__}: {str(exc)[:120]}")
            net = None
    va_x = va_y = None
    if net is None:
        tr_x, tr_y = collect_gt_crops(train_stems, "GT crop 수집 (train)")
        va_x, va_y = collect_gt_crops(valid_stems, "GT crop 수집 (val, 정확도 확인용)")
        print(f"train crops={len(tr_x):,}, val crops={len(va_x):,}, crop={SIMCLR_CROP}px, backbone={SIMCLR_BACKBONE}")
        net = train_simclr_classifier(tr_x, tr_y, va_x, va_y)
        net.eval()
        torch.save({
            "state_dict": net.state_dict(),
            "sorted_codes": sorted_codes,
            "backbone": SIMCLR_BACKBONE,
            "crop": SIMCLR_CROP,
        }, ckpt_path)
        print(f"저장: {ckpt_path}")
        del tr_x, tr_y
    if va_x is None:
        va_x, va_y = collect_gt_crops(valid_stems, "GT crop 수집 (val, 정확도 확인용)")
    acc = float((predict_probs(net, va_x, tta=SIMCLR_TTA).argmax(1) == va_y).mean())
    print(f"SimCLR 분류기 val GT-crop 정확도 (TTA={SIMCLR_TTA}): {acc:.4f}")
    simclr_net = net
    return net


# ── 검출 결과에 적용 ─────────────────────────────────────────
def simclr_probs_for_map(pred_map, stems, path_lookup, desc="SimCLR crop 분류"):
    """stem -> (K,C) 확률. 각 이미지에서 점수 상위 SIMCLR_TOPK개 중 점수 ≥ SIMCLR_MIN_SCORE인 선두 행만."""
    out, buf_crops, buf_keys = {}, [], []

    def flush():
        if not buf_crops:
            return
        probs = predict_probs(simclr_net, np.stack(buf_crops), tta=SIMCLR_TTA)
        for (stem_key, row), prob in zip(buf_keys, probs):
            out[stem_key][row] = prob
        buf_crops.clear()
        buf_keys.clear()

    for stem in tqdm(list(stems), desc=desc):
        pred = pred_map[stem]
        count = int((pred["scores"][:SIMCLR_TOPK] >= SIMCLR_MIN_SCORE).sum())
        out[stem] = np.zeros((count, num_classes), dtype=np.float32)
        if count == 0:
            continue
        with Image.open(path_lookup[stem]) as opened:
            image = opened.convert("RGB")
        for row in range(count):
            crop = crop_square(image, pred["xyxy"][row], SIMCLR_CROP, SIMCLR_CROP_PAD)
            buf_crops.append(np.asarray(crop))
            buf_keys.append((stem, row))
        image.close()
        if len(buf_crops) >= 256:
            flush()
    flush()
    return out


def fuse_simclr(pred, probs, alpha=None, topn=None):
    """검출 클래스 one-hot과 분류기 확률을 섞어 클래스 후보 top-N을 같은 박스에 출력.
    q = (1-alpha)*onehot(검출 클래스) + alpha*분류확률,  최종 점수 = 검출 점수 × q[class]"""
    alpha = SIMCLR_FUSE_ALPHA_FINAL if alpha is None else alpha
    topn = SIMCLR_TOPN if topn is None else topn
    boxes, scores, labels = [], [], []
    for i in range(len(pred["scores"])):
        if i < len(probs):
            q = alpha * probs[i]
            det_label = int(pred["labels"][i])
            if 0 <= det_label < num_classes:
                q[det_label] += 1.0 - alpha
            for cls in np.argsort(-q)[:topn]:
                if q[cls] <= 1e-6:
                    continue
                boxes.append(pred["xyxy"][i])
                scores.append(float(pred["scores"][i]) * float(q[cls]))
                labels.append(int(cls))
        else:
            boxes.append(pred["xyxy"][i])
            scores.append(float(pred["scores"][i]))
            labels.append(int(pred["labels"][i]))
    return make_pred(np.asarray(boxes).reshape(-1, 4), scores, labels, pred["wh"])


if USE_SIMCLR:
    try:
        setup_simclr()
        SIMCLR_ACTIVE = True
    except Exception:
        import traceback
        traceback.print_exc()
        print("⚠ SimCLR 준비에 실패해 SimCLR 없이 진행합니다. (위 오류를 확인하세요)")
        SIMCLR_ACTIVE = False
else:
    print("USE_SIMCLR=False: SimCLR 단계를 건너뜁니다.")


## 16. 변형별 val 점수 비교 · 제출 변형 결정


In [ ]:
# ══ 변형별 val 점수 비교 + 제출 변형 결정 ══
# base / sam / simclr / sam+simclr 중 켜져 있는 조합만 계산합니다.
val_maps = {"base": val_raw}
val_probs = {}

if SAM_ACTIVE:
    val_maps["sam"] = {s: apply_sam(val_raw[s], val_sam[s]) for s in valid_stems}

if SIMCLR_ACTIVE:
    try:
        val_probs["base"] = simclr_probs_for_map(val_raw, valid_stems, val_paths, desc="SimCLR crop 분류 (val, 원본 박스)")
        if SAM_ACTIVE:
            val_probs["sam"] = simclr_probs_for_map(val_maps["sam"], valid_stems, val_paths, desc="SimCLR crop 분류 (val, SAM 박스)")

        # 실제 제출에 쓰일 박스 기준으로 융합 비율(alpha) 선택
        tune_key = "sam" if SAM_ACTIVE else "base"
        if SIMCLR_TUNE_ALPHA:
            alpha_rows = []
            for alpha in SIMCLR_ALPHA_GRID:
                fused = {s: fuse_simclr(val_maps[tune_key][s], val_probs[tune_key][s], alpha=alpha) for s in valid_stems}
                alpha_rows.append({"alpha": float(alpha), **evaluate_variant(fused, valid_stems, val_targets)})
            alpha_table = pd.DataFrame(alpha_rows)
            display(alpha_table)
            SIMCLR_FUSE_ALPHA_FINAL = float(alpha_table.sort_values(["map8", "map20"], ascending=False).iloc[0]["alpha"])
            print(f"SimCLR 융합 alpha 자동 선택: {SIMCLR_FUSE_ALPHA_FINAL} (val 기준)")
        else:
            SIMCLR_FUSE_ALPHA_FINAL = float(SIMCLR_FUSE_ALPHA)

        val_maps["simclr"] = {s: fuse_simclr(val_raw[s], val_probs["base"][s]) for s in valid_stems}
        if SAM_ACTIVE:
            val_maps["sam+simclr"] = {s: fuse_simclr(val_maps["sam"][s], val_probs["sam"][s]) for s in valid_stems}
    except Exception:
        import traceback
        traceback.print_exc()
        print("⚠ SimCLR 융합 평가 중 오류 → SimCLR 없이 진행합니다.")
        SIMCLR_ACTIVE = False
        val_maps.pop("simclr", None)
        val_maps.pop("sam+simclr", None)

for name, pred_map in val_maps.items():
    if name not in variant_scores:
        variant_scores[name] = evaluate_variant(pred_map, valid_stems, val_targets)

comparison = pd.DataFrame(variant_scores).T.rename(columns={
    "map20": f"mAP@[.75:.95] (max {EVAL_MAX_DETECTIONS})",
    "map8": f"mAP@[.75:.95] (max {SUBMISSION_MAX_DETECTIONS}, 제출 조건)",
})
comparison["Δ vs base (제출 조건)"] = (
    comparison[f"mAP@[.75:.95] (max {SUBMISSION_MAX_DETECTIONS}, 제출 조건)"]
    - comparison.loc["base", f"mAP@[.75:.95] (max {SUBMISSION_MAX_DETECTIONS}, 제출 조건)"]
)
display(comparison.style.format("{:.5f}"))

FINAL_VARIANT = "+".join(
    name for name, on in (("sam", SAM_ACTIVE), ("simclr", SIMCLR_ACTIVE)) if on
) or "base"
if SUBMIT_USE_BEST_VARIANT:
    FINAL_VARIANT = max(variant_scores, key=lambda k: (variant_scores[k]["map8"], variant_scores[k]["map20"]))
print(f"USE_SAM={USE_SAM} (실제 적용={SAM_ACTIVE}) | USE_SIMCLR={USE_SIMCLR} (실제 적용={SIMCLR_ACTIVE})")
print(f"제출에 사용할 변형: {FINAL_VARIANT}  (val {variant_scores[FINAL_VARIANT]['map8']:.5f})")
if FINAL_VARIANT != "base" and variant_scores[FINAL_VARIANT]["map8"] < variant_scores["base"]["map8"]:
    print("⚠ 선택된 변형의 val 점수가 base보다 낮습니다. 스위치를 끄거나 SUBMIT_USE_BEST_VARIANT=True를 고려하세요.")


## 14. validation 예측 시각화


In [ ]:
def show_prediction(stem: str, threshold: float = 0.20, figsize=(10, 8)) -> None:
    image = Image.open(image_records[stem]["image_path"]).convert("RGB")
    detections = best_model.predict(image, threshold=threshold)
    order = np.argsort(-detections.confidence)[:EVAL_MAX_DETECTIONS]
    detections = detections[order]
    labels = [
        f"{model_id_to_code[int(class_id)]} {score:.3f}"
        for class_id, score in zip(detections.class_id, detections.confidence)
    ]
    canvas = np.asarray(image).copy()
    canvas = sv.BoxAnnotator(thickness=3).annotate(canvas, detections)
    canvas = sv.LabelAnnotator(text_scale=0.45).annotate(canvas, detections, labels)
    plt.figure(figsize=figsize)
    plt.imshow(canvas)
    plt.title(f"Prediction · {stem}")
    plt.axis("off")
    plt.show()
    image.close()


for sample_stem in random.sample(valid_stems, min(4, len(valid_stems))):
    show_prediction(sample_stem)


## 17. 테스트 추론 · 제출 CSV 생성

제출 컬럼: `annotation_id,image_id,category_id,bbox_x,bbox_y,bbox_w,bbox_h,score`


In [ ]:
# ══ 테스트 추론 → 변형 적용 → 제출 CSV ══
SUBMISSION_PATH = RUN_OUTPUT_DIR / f"submission_{SUBMISSION_TAG}_{FINAL_VARIANT.replace('+', '_')}.csv"

test_lookup = {p.stem: p for p in test_paths}
test_stems = [p.stem for p in test_paths]

test_raw = {}
for path_batch in tqdm(list(batched(test_paths, PRED_BATCH_SIZE)), desc="Test 검출"):
    for path, pred in zip(path_batch, detect_batch(path_batch, max_det=EVAL_MAX_DETECTIONS)):
        test_raw[path.stem] = pred

final_parts = FINAL_VARIANT.split("+")
test_pred = test_raw
if "sam" in final_parts:
    test_pred = {
        s: apply_sam(test_raw[s], sam_boxes_for_pred(test_lookup[s], test_raw[s]))
        for s in tqdm(test_stems, desc="SAM 정제 (test)")
    }
if "simclr" in final_parts:
    test_probs = simclr_probs_for_map(test_pred, test_stems, test_lookup, desc="SimCLR crop 분류 (test)")
    test_pred = {s: fuse_simclr(test_pred[s], test_probs[s]) for s in test_stems}

rows, annotation_id = [], 1
for stem in test_stems:
    try:
        image_id = int(stem)
    except ValueError as exc:
        raise ValueError(f"test filename stem이 숫자가 아닙니다: {stem!r}") from exc
    pred = test_pred[stem]
    boxes = clip_xyxy(pred["xyxy"], pred["wh"])
    for i in range(min(len(pred["scores"]), SUBMISSION_MAX_DETECTIONS)):
        x1, y1, x2, y2 = [float(v) for v in boxes[i]]
        bw, bh = x2 - x1, y2 - y1
        class_id = int(pred["labels"][i])
        if bw <= 0 or bh <= 0 or not (0 <= class_id < num_classes):
            continue
        rows.append({
            "annotation_id": annotation_id,
            "image_id": image_id,
            "category_id": code_to_submission_id[model_id_to_code[class_id]],
            "bbox_x": round(x1, 4),
            "bbox_y": round(y1, 4),
            "bbox_w": round(bw, 4),
            "bbox_h": round(bh, 4),
            "score": round(float(min(max(pred["scores"][i], 0.0), 1.0)), 6),
        })
        annotation_id += 1

submission_columns = ["annotation_id", "image_id", "category_id", "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score"]
submission = pd.DataFrame(rows, columns=submission_columns)
submission.to_csv(SUBMISSION_PATH, index=False, encoding="utf-8")
print(f"변형: {FINAL_VARIANT}")
print(f"저장 완료: {SUBMISSION_PATH.resolve()} / rows={len(submission):,}")
display(submission.head())


## 18. 제출 파일 검증


In [ ]:
assert submission.columns.tolist() == submission_columns
assert submission["annotation_id"].is_unique
assert (submission["bbox_w"] > 0).all()
assert (submission["bbox_h"] > 0).all()
assert submission["score"].between(0, 1).all()
assert set(submission["category_id"]).issubset(set(code_to_submission_id.values()))

test_image_ids = {int(path.stem) for path in test_paths}
prediction_counts = submission.groupby("image_id").size()
missing_prediction_ids = test_image_ids - set(prediction_counts.index)

print(f"제출 대상 이미지 수   : {len(test_image_ids):,}")
print(f"예측 포함 이미지 수   : {prediction_counts.size:,}")
print(f"예측 0개 test 이미지 : {len(missing_prediction_ids):,}")
print("이미지당 예측 수 분포:")
display(prediction_counts.describe())

if missing_prediction_ids:
    raise RuntimeError(
        "예측이 0개인 test 이미지가 있습니다: "
        f"{sorted(missing_prediction_ids)[:20]}"
    )


## 운영 메모 (v2)

1. 처음에는 `USE_SAM=False, USE_SIMCLR=False`로 기준선(base)을 만들고, 그 다음 스위치를 하나씩 켜서 비교표를 확인하세요.
2. **먼저 560으로 완주**하세요. 물리 배치는 AutoBatch 결과를 기록해 두세요.
3. `ACCEPT_PLATFORM_MODEL_LICENSE`는 PML-1.0을 직접 검토하고 동의한 경우에만 True로 바꾸세요.
4. 모델 선택에는 기본 COCO mAP가 아니라 비교표의 제출 조건(max 8) 열을 사용하세요.
5. 증강 family 분포가 예상과 다르면(경고 출력) 파일명의 원본 ID 규칙으로 `augmentation_family_signature()`를 교체하는 것을 검토하세요.
6. SAM/SimCLR는 val로 스칼라(padding, alpha)를 튜닝하므로 val 점수가 약간 낙관적일 수 있습니다.

공식 문서:

- RF-DETR+ repository: <https://github.com/roboflow/rf-detr_plus>
- Training parameters: <https://rfdetr.roboflow.com/latest/learn/train/training-parameters/>
- Dataset formats: <https://rfdetr.roboflow.com/latest/learn/train/dataset-formats/>
